# 04 — Linear + Knowledge Store (read-only) + runtime probes

Three surfaces:

1. **Runtime selection probe** — render `HSB_RUNTIME_*` and the Codex-availability check (no LLM, no MCP).
2. **Linear MCP — read-only.** Confirms wiring + hooks behaviour. Gated on `HSB_NOTEBOOK_RUN_LIVE=1`. Token-cheap because everything we issue is a `read` operation; no writes mean the G5 `linear_write_guard` doesn't matter here.
3. **Knowledge Store — pure filesystem retrieval.** Glob + Grep over `knowledge/` mirrors what the Intelligence Agent does inline in the WIO (skill 10). No LLM, no MCP, no runtime.

The Linear cell drives `run_validated_linear_agent`, which goes through whichever runtime `HSB_RUNTIME_LINEAR` selects. With `HSB_RUNTIME_LINEAR=codex` you must also have a valid `~/.codex/config.toml` with the Linear MCP server registered — `codex_available()` and `verify_codex_mcp()` flag a misconfig before the call burns tokens.

If Linear cells skip on first run that's fine — the Knowledge Store cells run regardless and are the higher-value half.

In [ ]:
import os

from _helpers import (
    assert_g1_safe,
    codex_available,
    ensure_src_on_path,
    gated,
    live_mode,
    runtime_summary,
    selected_runtime,
)

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_* selection:\n" + runtime_summary())
ok, reason = codex_available()
print(f"\ncodex_available -> ok={ok}  reason={reason}")

## Knowledge Store — inventory

Categories (per `README.md#repository-layout`): `architecture`, `qa`, `implementation`, `backlog`, `risk`, `patterns`, `anti-patterns`.

In [ ]:
ks = ROOT / "knowledge"
if not ks.exists():
    print("(knowledge/ not present at repo root — nothing to enumerate)")
else:
    for cat in sorted(p.name for p in ks.iterdir() if p.is_dir()):
        entries = list((ks / cat).glob("*.md"))
        print(f"{cat:>15}  {len(entries):>3} entries")

## Knowledge Store — Glob + Grep retrieval (skill 10 surface)

Equivalent to what `build_enrichment_prompt` instructs the LLM to run: list candidate files, then grep them for keywords matching the work item's domain / technology / pattern.

Usage: edit `keywords` to whatever your task is about, run, see what entries the Intelligence Agent would surface.

In [ ]:
import re

if not ks.exists():
    print("(knowledge/ not present — skipping grep)")
else:
    keywords = ["postgres", "pydantic", "retry", "optimistic"]
    pat = re.compile("|".join(re.escape(k) for k in keywords), re.I)
    hits = []
    for p in ks.rglob("*.md"):
        text = p.read_text(errors="ignore")
        matches = pat.findall(text)
        if matches:
            hits.append((p.relative_to(ROOT), len(matches)))
    for path, n in sorted(hits, key=lambda x: -x[1]):
        print(f"{n:>4}  {path}")
    if not hits:
        print(
            "(no entries match — knowledge/ may be empty on main; populated by skill 11 over time)"
        )

## Knowledge Store — write a transient fixture entry

Drops one entry into `knowledge/qa/` under a tmp filename so you can re-run the grep cell and see it ranked, then deletes it. Mirrors what `KnowledgeStorageInput` would write.

In [ ]:
import datetime

from hsb.contracts.knowledge import KnowledgeStorageInput

if not ks.exists():
    print("(knowledge/ not present — skipping fixture)")
else:
    entry = KnowledgeStorageInput(
        title="Postgres optimistic-lock retry pattern",
        type="qa",
        context="Linear writes were silently overwritten by concurrent claims",
        evidence={
            "linear_issue": "LIN-99",
            "pr": "#42",
            "files": ["linear_agent.py"],
            "qa_finding": "F-7",
        },
        insight="updatedAt re-read is the only way to detect a stale write",
        recommendation="read -> write -> re-read; verify post > pre, not equality",
        applicability="Any Linear write path during parallel claiming",
        date=datetime.date.today().isoformat(),
    )
    qa_dir = ks / "qa"
    qa_dir.mkdir(parents=True, exist_ok=True)
    tmp = qa_dir / f"_nb-fixture-{os.getpid()}.md"
    tmp.write_text(
        f"---\ntitle: {entry.title}\ntype: {entry.type}\napplicability: {entry.applicability}\ndate: {entry.date}\n---\n\n"
        f"## Insight\n{entry.insight}\n\n## Recommendation\n{entry.recommendation}\n"
    )
    try:
        print("wrote fixture:", tmp.relative_to(ROOT))
        pat2 = re.compile("postgres|optimistic", re.I)
        matches = [
            p for p in ks.rglob("*.md") if pat2.search(p.read_text(errors="ignore"))
        ]
        assert tmp in matches, "fixture not picked up by grep"
        print(f"grep retrieves {len(matches)} entries including the fixture")
    finally:
        tmp.unlink(missing_ok=True)
        print("fixture removed")

## Linear Agent — read probes (gated)

Two reads: `list_teams` and a single-issue `get_issue` (only if `HSB_NOTEBOOK_LINEAR_ISSUE_ID` is set). Both go through `run_validated_linear_agent` so the validation/retry loop is exercised. `HSB_RUNTIME_LINEAR` selects which runtime executes the call.

Gates:
- `HSB_NOTEBOOK_RUN_LIVE=1` — opt in to actual SDK calls.
- For `HSB_RUNTIME_LINEAR=claude`: `CLAUDE_CODE_OAUTH_TOKEN` must be set.
- For `HSB_RUNTIME_LINEAR=codex`: `~/.codex/config.toml` with `forced_login_method = "chatgpt"` and a `[mcp_servers.linear]` block, plus `~/.codex/auth.json` (run `codex login --device-auth`).

First-run warning (Claude side): `mcp-remote` to `mcp.linear.app/mcp` triggers an interactive OAuth flow. Run the smoke test in `linear_agent.py` once from the CLI before relying on the notebook.

In [ ]:
if not live_mode():
    print(gated("Linear list_teams"))
else:
    assert_g1_safe()
    rt = selected_runtime("linear")
    print(f"(running on HSB_RUNTIME_LINEAR={rt!r})")
    if rt == "codex":
        ok, reason = codex_available()
        if not ok:
            print(f"[skipped] codex selected but unavailable: {reason}")
        else:
            print("(codex config OK; proceeding)")
    import asyncio

    from hsb.agents.linear_agent import run_validated_linear_agent

    out = asyncio.run(
        run_validated_linear_agent(
            operation="read",
            payload={"kind": "teams"},
        )
    )
    print("result =", out.result, "| entities =", len(out.linear_entities))

In [ ]:
issue_id = os.environ.get("HSB_NOTEBOOK_LINEAR_ISSUE_ID")
if not live_mode() or not issue_id:
    print(gated("Linear get_issue (set HSB_NOTEBOOK_LINEAR_ISSUE_ID=LIN-...)"))
else:
    assert_g1_safe()
    import asyncio

    from hsb.agents.linear_agent import run_validated_linear_agent

    out = asyncio.run(
        run_validated_linear_agent(
            operation="read",
            payload={"issueId": issue_id},
        )
    )
    print("result =", out.result, "| entities =", [e.id for e in out.linear_entities])

## Linear hooks — module-level inspection

The hooks run inside the SDK loop, but the per-tool retry counter and audit log path are public module state we can sanity-check. Hooks are Claude-only (the HookMatcher API has no Codex equivalent — see `CodexRuntime.query` which raises if `options.hooks is not None`).

In [ ]:
from hsb.agents import hooks

print("MAX_RETRIES =", hooks.MAX_RETRIES)
print("BASE_DELAY_SECONDS =", hooks.BASE_DELAY_SECONDS)
print("AUDIT_LOG_PATH =", hooks.AUDIT_LOG_PATH)
print(
    "LINEAR_HOOKS keys =",
    list(hooks.LINEAR_HOOKS.keys()) if hasattr(hooks, "LINEAR_HOOKS") else "n/a",
)